In [6]:
# imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
from sklearn import tree

In [7]:
# data load
base_data = pd.read_csv("./fsae_data/2025/raw-data.csv")

cost_data = pd.read_csv("./fsae_data/2025/fsae_cost_results.csv")
cost_data = cost_data.drop(columns=["Place", "Car Num", "Penalty", "Score", "Penalty"])
merged_data = pd.merge(base_data, cost_data, on = 'Team', how = "outer")

efficiency_data = pd.read_csv("./fsae_data/2025/fsae_efficiency_results.csv")
efficiency_data = efficiency_data.drop(columns = ["Place", "Car Num", "Fuel Efficiency Score"])
merged_data = pd.merge(merged_data, efficiency_data, on= 'Team', how = 'outer')

presentation_data = pd.read_csv("./fsae_data/2025/fsae_presentation_results.csv")
presentation_data = presentation_data.drop(columns = ["Place", "Car Num", "Score", "Penalty", "Raw Score", "Status"])
merged_data = pd.merge(merged_data, presentation_data, on= 'Team', how = 'outer')

design_data = pd.read_csv("./fsae_data/2025/fsae_presentation_results.csv")
design_data = design_data.drop(columns = ["Place", "Car Num", "Score", "Penalty", "Raw Score", "Status"])
merged_data = pd.merge(merged_data, design_data, on= 'Team', how = 'outer')

accel_data = pd.read_csv("./fsae_data/2025/fsae_accel_results.csv", index_col=False)
accel_data = accel_data[["Team", "Final Best Time"]]
accel_data = accel_data.rename(columns = {"Final Best Time": "Best Acceleration Time"})
#accel_data = accel_data.drop(columns = ["Place", "Car Num", "Run 1 Time", "Run 1 Adj Time", "Run 2 Time", "Run 2 Adj Time", "Run 3 Time", "Run 3 Adj Time", "Run 4 Time", "Run 4 Adj Time"])
merged_data = pd.merge(merged_data, accel_data, on= 'Team', how = 'outer')

autocross_data = pd.read_csv("./fsae_data/2025/fsae_autocross_results.csv", index_col=False)
autocross_data = autocross_data[["Team", "Best Time"]]
autocross_data = autocross_data.rename(columns = {"Best Time": "Best Autocross Time"})
#autocross_data = autocross_data.drop(columns = ["Place", "Car Num", "Penalty","R1 Time","R1 Cones","R1 Off Course","R1 Adj Time","R2 Time","R2 Cones","R2 Off Course","R2 Adj Time","R3 Time","R3 Cones","R3 Off Course","R3 Adj Time","R4 Time","R4 Cones","R4 Off Course","R4 Adj Time"])
merged_data = pd.merge(merged_data, autocross_data, on= 'Team', how = 'outer')

endurance_data = pd.read_csv("./fsae_data/2025/fsae_endurance_results.csv", index_col=False)
endurance_data = endurance_data.drop(columns = ["Place", "Car Num", "Cones", "Endurance Score"])
endurance_data = endurance_data.rename(columns = {"Time": "Endurance Time", "Other Penalty": "Endurance Penalty", "Adjusted Time": "Adjusted Endurance Time", "Time Score": "Endurance Time Score", "Laps": "Endurance Laps", "Laps Score": "Endurance Laps Score"})
merged_data = pd.merge(merged_data, endurance_data, on= 'Team', how = 'outer')

skidpad_data = pd.read_csv("./fsae_data/2025/fsae_skidpad_results.csv", index_col=False)
skidpad_data = skidpad_data[["Team", "Best Time"]]
skidpad_data = skidpad_data.rename(columns = {"Best Time": "Best Skidpad Time"})
merged_data = pd.merge(merged_data, skidpad_data, on= 'Team', how = 'outer')


team_data = pd.read_csv("./fsae_data/2025/fsae_team_info.csv", index_col=False)
team_data = team_data[["Team", "Engine Cylinders", "Engine Displacement (cc)", "Weight (kg)"]]
merged_data = pd.merge(merged_data, skidpad_data, on= 'Team', how = 'outer')

print(merged_data)

        Place  Car Num                                 Team  Penalty  \
0    Unranked    135.0                   Alabama A & M Univ      NaN   
1          75     49.0           Arizona State Univ - Tempe      NaN   
2          18     30.0                           Brown Univ      NaN   
3          33    148.0                             CEFET-MG      NaN   
4          39     89.0  California State Poly Univ - Pomona      NaN   
..        ...      ...                                  ...      ...   
121        58     97.0                     Wayne State Univ      NaN   
122        73     66.0                   West Virginia Univ      NaN   
123        68    132.0                   Wichita State Univ      NaN   
124        92     62.0         York College of Pennsylvania      NaN   
125         5    106.0                      lowa State Univ      NaN   

     Cost Score  Presentation Score  Design Score  Acceleration Score  \
0          18.5                 NaN           NaN             

In [8]:
# Clean place data
placement_new = pd.DataFrame.copy(merged_data)
placement_new = placement_new.dropna(subset=["Place"])
unranked = merged_data[merged_data["Place"] == "Unranked"]
placement_new = placement_new[placement_new.Place != "Unranked"]

tied_cases = placement_new["Place"].str.contains('T')
#tied_cases = tied_cases.fillna(False)
tied = placement_new[tied_cases]
tied["Place"] = tied["Place"].str[:-2]
placement_new = placement_new[~tied_cases]
placement_new = pd.concat([placement_new, tied], ignore_index = True)

placement_new["Place"] = placement_new["Place"].astype(int)
placement_new = placement_new.reset_index(drop = True)
adjustment = pd.DataFrame.copy(placement_new)
for i in range(len(placement_new)):
    if placement_new.iloc[i]["Place"] == 1:
        adjustment.at[i,"Place"] = "1"
    elif placement_new.iloc[i]["Place"] <= 5:
        adjustment.at[i,"Place"] = "<=5"
    elif placement_new.iloc[i]["Place"] <= 10:
        adjustment.at[i,"Place"] = "<=10"
    elif placement_new.iloc[i]["Place"] <= 25:
        adjustment.at[i,"Place"] = "<=25"
    elif placement_new.iloc[i]["Place"] <= 50:
        adjustment.at[i,"Place"] = "<=50"
    elif placement_new.iloc[i]["Place"] <= 75:
        adjustment.at[i,"Place"] = "<=75"
    elif placement_new.iloc[i]["Place"] <= 100:
        adjustment.at[i,"Place"] = "<=100"
    else:
        adjustment.at[i,"Place"] = ">100"

placement_new["Place"] = adjustment["Place"]
with pd.option_context('display.max_rows', None):
    print(placement_new)

     Place  Car Num                                      Team  Penalty  \
0     <=75     49.0                Arizona State Univ - Tempe      NaN   
1     <=25     30.0                                Brown Univ      NaN   
2     <=50    148.0                                  CEFET-MG      NaN   
3     <=50     89.0       California State Poly Univ - Pomona      NaN   
4    <=100     67.0         California State Univ - Fullerton      NaN   
5     <=25     84.0        California State Univ - Northridge      NaN   
6    <=100     55.0                             Carleton Univ      NaN   
7     <=75     50.0                       Clarkson University      NaN   
8     <=50     58.0                              Clemson Univ      NaN   
9    <=100     99.0                  Colorado Mesa University      NaN   
10    <=50     23.0                  Colorado School of Mines      NaN   
11    <=75     44.0                 Colorado State University      NaN   
12    <=25     34.0                   

C:\Users\gsant\AppData\Local\Temp\ipykernel_34524\3059716796.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tied["Place"] = tied["Place"].str[:-2]
C:\Users\gsant\AppData\Local\Temp\ipykernel_34524\3059716796.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<=75' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  adjustment.at[i,"Place"] = "<=75"


In [9]:
# acceleration decision tree


In [10]:
clean_data = pd.DataFrame.copy(placement_new)